# Pothole Detection for Malaysian Road Maintenance with YOLOv8

**Problem:** Potholes are a constant public complaint on Malaysian roads, and local councils and JKR still depend largely on manual inspection and public reports to find them. This project trains an object detection model that automatically locates potholes in road images, so it could run on dashcams or roadside edge devices to map road damage.

**Dataset:** Pothole Detection dataset (Kaggle: `andrewmvd/pothole-detection`), ~665 road images with bounding-box labels (1 class: pothole).

**Model:** YOLOv8s (Ultralytics), fine-tuned from COCO-pretrained weights.

**Metrics:** mAP@0.5, mAP@0.5:0.95, precision, recall.

> Before running: go to **Runtime → Change runtime type → T4 GPU**.

## Step 1: Install libraries and check the GPU

In [ ]:
!pip install -q ultralytics kagglehub
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None - change runtime to T4 GPU!")

## Step 2: Download the dataset from Kaggle

In [ ]:
import kagglehub
raw_path = kagglehub.dataset_download("andrewmvd/pothole-detection")
print("Dataset downloaded to:", raw_path)

## Step 3: Explore the dataset (count images and annotations per class)

In [ ]:
import os, glob
from collections import Counter
import xml.etree.ElementTree as ET

xml_files = glob.glob(os.path.join(raw_path, "**", "*.xml"), recursive=True)
img_files = [f for ext in ("jpg", "jpeg", "png", "JPG")
             for f in glob.glob(os.path.join(raw_path, "**", f"*.{ext}"), recursive=True)]
print("Annotation files:", len(xml_files))
print("Image files:", len(img_files))

box_counts = Counter()
for x in xml_files:
    for obj in ET.parse(x).getroot().findall("object"):
        box_counts[obj.find("name").text.strip().lower()] += 1

CLASSES = sorted(box_counts)
print("\nClasses:", CLASSES)
print("\nBounding boxes per class:")
for c in CLASSES:
    print(f"  {c:18s} {box_counts[c]}")

## Step 4: Convert Pascal VOC (XML) labels to YOLO format and split 70/15/15

In [ ]:
import random, shutil

random.seed(42)  # fixed seed so the split is reproducible
OUT = "/content/pothole_yolo"
shutil.rmtree(OUT, ignore_errors=True)

# map image file name (without extension) -> full path
img_lookup = {os.path.splitext(os.path.basename(p))[0]: p for p in img_files}

def voc_to_yolo(xml_path):
    root = ET.parse(xml_path).getroot()
    w = float(root.find("size/width").text)
    h = float(root.find("size/height").text)
    lines = []
    for obj in root.findall("object"):
        cls = CLASSES.index(obj.find("name").text.strip().lower())
        b = obj.find("bndbox")
        xmin, ymin = float(b.find("xmin").text), float(b.find("ymin").text)
        xmax, ymax = float(b.find("xmax").text), float(b.find("ymax").text)
        xc, yc = (xmin + xmax) / 2 / w, (ymin + ymax) / 2 / h
        bw, bh = (xmax - xmin) / w, (ymax - ymin) / h
        lines.append(f"{cls} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
    return lines

pairs = []
for x in xml_files:
    stem = os.path.splitext(os.path.basename(x))[0]
    if stem in img_lookup:
        pairs.append((img_lookup[stem], x))
print("Matched image-label pairs:", len(pairs))

random.shuffle(pairs)
n = len(pairs)
splits = {"train": pairs[:int(0.7*n)],
          "val":   pairs[int(0.7*n):int(0.85*n)],
          "test":  pairs[int(0.85*n):]}

for split, items in splits.items():
    os.makedirs(f"{OUT}/images/{split}", exist_ok=True)
    os.makedirs(f"{OUT}/labels/{split}", exist_ok=True)
    for img, x in items:
        stem = os.path.splitext(os.path.basename(img))[0]
        shutil.copy(img, f"{OUT}/images/{split}/{os.path.basename(img)}")
        with open(f"{OUT}/labels/{split}/{stem}.txt", "w") as f:
            f.write("\n".join(voc_to_yolo(x)))
    print(f"{split}: {len(items)} images")

# dataset config file for YOLO
with open(f"{OUT}/data.yaml", "w") as f:
    f.write(f"path: {OUT}\ntrain: images/train\nval: images/val\ntest: images/test\n")
    f.write(f"nc: {len(CLASSES)}\nnames: {CLASSES}\n")
print(open(f"{OUT}/data.yaml").read())

## Step 5: Visualise one labelled training image (sanity check)

In [ ]:
import cv2, matplotlib.pyplot as plt

sample = sorted(os.listdir(f"{OUT}/images/train"))[0]
img = cv2.cvtColor(cv2.imread(f"{OUT}/images/train/{sample}"), cv2.COLOR_BGR2RGB)
H, W = img.shape[:2]
for line in open(f"{OUT}/labels/train/{os.path.splitext(sample)[0]}.txt"):
    c, xc, yc, bw, bh = map(float, line.split())
    x1, y1 = int((xc-bw/2)*W), int((yc-bh/2)*H)
    x2, y2 = int((xc+bw/2)*W), int((yc+bh/2)*H)
    cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 3)
    cv2.putText(img, CLASSES[int(c)], (x1, y1-15), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
plt.figure(figsize=(12, 8)); plt.imshow(img); plt.axis("off"); plt.title(sample); plt.show()

## Step 6: Train YOLOv8s

Transfer learning: we start from weights pretrained on COCO and fine-tune on potholes. On a free T4 GPU this takes roughly 15 to 30 minutes. If you get an out-of-memory error, change `batch=16` to `batch=8`.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")   # COCO-pretrained weights (transfer learning)
results = model.train(
    data=f"{OUT}/data.yaml",
    epochs=60,
    imgsz=640,
    batch=16,
    patience=15,     # early stopping if validation mAP stops improving
    seed=42,
    project="/content/runs",
    name="pothole_yolov8s",
    exist_ok=True,
)

## Step 7: Training curves

In [ ]:
from IPython.display import Image, display
RUN = "/content/runs/pothole_yolov8s"
display(Image(f"{RUN}/results.png", width=1000))

## Step 8: Evaluate on the held-out TEST set (use these numbers in your form/CV)

In [ ]:
best = YOLO(f"{RUN}/weights/best.pt")
metrics = best.val(data=f"{OUT}/data.yaml", split="test", imgsz=640, batch=16,
                   project="/content/runs", name="pothole_test", exist_ok=True)

print("\n===== TEST SET RESULTS =====")
print(f"mAP@0.5      : {metrics.box.map50:.3f}")
print(f"mAP@0.5:0.95 : {metrics.box.map:.3f}")
print(f"Precision    : {metrics.box.mp:.3f}")
print(f"Recall       : {metrics.box.mr:.3f}")
print("\nmAP@0.5 per class:")
for i, c in enumerate(CLASSES):
    print(f"  {c:18s} {metrics.box.ap50[i]:.3f}")

## Step 9: Confusion matrix

In [ ]:
display(Image("/content/runs/pothole_test/confusion_matrix_normalized.png", width=700))

## Step 10: Predictions on unseen test images

In [ ]:
test_imgs = sorted(glob.glob(f"{OUT}/images/test/*"))[:4]
preds = best.predict(test_imgs, imgsz=640, conf=0.25, verbose=False)
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
for ax, p in zip(axes.ravel(), preds):
    ax.imshow(cv2.cvtColor(p.plot(line_width=3), cv2.COLOR_BGR2RGB))
    ax.set_title(os.path.basename(p.path)); ax.axis("off")
plt.tight_layout()
plt.savefig("/content/sample_predictions.png", dpi=100)
plt.show()

## Step 11 (recommended): Test on YOUR OWN Malaysian road photos

Take a few photos of potholes on local roads with your phone (or screenshots from Google Street View in Malaysia), then run this cell and upload them. This shows how well the model generalises to Malaysian roads, which is a great talking point in an interview.

In [ ]:
from google.colab import files
uploaded = files.upload()
my_preds = best.predict(list(uploaded.keys()), imgsz=640, conf=0.25, verbose=False)
for p in my_preds:
    plt.figure(figsize=(10, 7))
    plt.imshow(cv2.cvtColor(p.plot(line_width=3), cv2.COLOR_BGR2RGB))
    plt.title(f"{os.path.basename(p.path)}: {len(p.boxes)} pothole(s) detected"); plt.axis("off")
    plt.savefig(f"/content/malaysia_{os.path.splitext(os.path.basename(p.path))[0]}.png", dpi=100)
    plt.show()

## Step 12: Download results for your GitHub repo

In [ ]:
import shutil
os.makedirs("/content/results_for_github", exist_ok=True)
for f in [f"{RUN}/results.png",
          "/content/runs/pothole_test/confusion_matrix_normalized.png",
          "/content/sample_predictions.png",
          *glob.glob("/content/malaysia_*.png"),
          f"{RUN}/weights/best.pt"]:
    shutil.copy(f, "/content/results_for_github/")
shutil.make_archive("/content/results_for_github", "zip", "/content/results_for_github")

from google.colab import files
files.download("/content/results_for_github.zip")

## Step 13: Turn the model into a user-friendly web app (Gradio)

Run this cell, then click the **public link** (ends with `gradio.live`) that appears. Anyone can open it on a phone or laptop, upload a road photo and see the potholes detected. The link works for 72 hours while this Colab session is running, which is enough for a demo or a screenshot for your README.

In [ ]:
!pip install -q gradio
import gradio as gr
import numpy as np

app_model = YOLO(f"{RUN}/weights/best.pt")

def detect_potholes(image, confidence):
    if image is None:
        return None, "Please upload a road photo."
    result = app_model.predict(image[:, :, ::-1], imgsz=640, conf=confidence, verbose=False)[0]
    annotated = result.plot(line_width=3)[:, :, ::-1]  # BGR -> RGB
    n = len(result.boxes)
    if n == 0:
        summary = "No potholes detected. Road looks OK."
    else:
        h, w = result.orig_shape
        areas = [(float(b[2]-b[0]) * float(b[3]-b[1])) / (w*h) * 100 for b in result.boxes.xyxy]
        severity = "HIGH" if n >= 3 or max(areas) > 5 else "MEDIUM" if n == 2 or max(areas) > 1.5 else "LOW"
        summary = (f"Potholes detected: {n}\n"
                   f"Largest pothole: {max(areas):.1f}% of image\n"
                   f"Suggested repair priority: {severity}")
    return annotated, summary

demo = gr.Interface(
    fn=detect_potholes,
    inputs=[gr.Image(type="numpy", label="Upload a road photo"),
            gr.Slider(0.1, 0.9, value=0.25, step=0.05, label="Detection confidence threshold")],
    outputs=[gr.Image(label="Detected potholes"),
             gr.Textbox(label="Result", lines=3)],
    title="Pothole Detector for Malaysian Roads",
    description="Upload a photo of a road. The YOLOv8 model marks each pothole and suggests a repair priority to help councils plan maintenance.",
    flagging_mode="never",
)
demo.launch(share=True)